**Curated source notebook — archival code, not validated as end-to-end runnable.**\n\nExecution outputs and environment metadata were removed for repository hygiene. Original computational claims require independent validation; see `docs/AUDIT_STATUS.md`.\n

# 0- Connect to the GPU. (runtime --> change runtime type --> select T4 GPU)

# 1- connect to your google drive (you need at least 10 GB empty space on your drive. If you do not have enough space, create a new gmail and use it)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 2- build GROMACS (this will take a long time. approximately 1 hour)<br> DO NOT CLOSE THE TAB. <br>BE NEAR THE COMPUTER, TO RECONNECT IF THE SESSION CLOSED SUDDENLY <br> **(THIS IS DONE ONE TIME FOR THE SAME GMAIL ACCOUNT)**

In [ ]:
!mkdir /content/drive/MyDrive/GROMACS_GPU

In [ ]:
%%shell
wget ftp://ftp.gromacs.org/gromacs/gromacs-2024.4.tar.gz
tar xfz gromacs-2024.4.tar.gz
cd gromacs-2024.4
mkdir build
cd build
cmake .. -DGMX_BUILD_OWN_FFTW=ON -DREGRESSIONTEST_DOWNLOAD=ON -DCMAKE_INSTALL_PREFIX=/content/drive/MyDrive/GROMACS_GPU -DGMX_GPU=CUDA
make -j 2
make check -j 2
sudo make install
chmod +x /content/drive/MyDrive/GROMACS_GPU/bin/*
source /content/drive/MyDrive/GROMACS_GPU/bin/GMXRC
gmx

# 3- Make sure GROMACS runs correctly

In [ ]:
%%shell
chmod +x /content/drive/MyDrive/GROMACS_GPU/bin/*
source /content/drive/MyDrive/GROMACS_GPU/bin/GMXRC
gmx

# If You have an account on CHARMM-GUI,<br> prepare the system on the website <br> then uplaod the .zip on the drive inside a folder that will have simulation data <br>then start the next step

# 4- Start minimization

* Download the .tgz file from CHARMM-GUI webserver and extract it.
* Compress the gromacs folder as .zip file and upload it on Drive in a specific folder
* Change the \<PATH_MAIN> (in the next cell) to the folder that contain the .zip file
* Change the \<FILE> (in the next cell) to the name of the .zip file


In [ ]:
%%shell
cd /content/drive/MyDrive/GROMACS_BIOCODERS
unzip gromacs.zip

* Download .mdp files (run the next cell)


In [ ]:
# [REDACTED FILE ID] Obtain the input from the original project workspace before running.
# [REDACTED FILE ID] Obtain the input from the original project workspace before running.
# [REDACTED FILE ID] Obtain the input from the original project workspace before running.
# [REDACTED FILE ID] Obtain the input from the original project workspace before running.
!mv *.mdp /content/drive/MyDrive/GROMACS_BIOCODERS/gromacs

* change \<PATH> ***in ALL the next cells*** to the **gromacs** folder obtained from extracting the .zip file

In [ ]:
!mv *.mdp /content/drive/MyDrive/GROMACS_BIOCODERS/gromacs

In [ ]:
%%shell
chmod +x /content/drive/MyDrive/GROMACS_GPU/bin/*
source /content/drive/MyDrive/GROMACS_GPU/bin/GMXRC
cd /content/drive/MyDrive/GROMACS_BIOCODERS/gromacs

gmx grompp -f step4.0_minimization.mdp -c step3_input.gro -p topol.top -o step4.0_minimization.tpr
gmx mdrun -v -deffnm step4.0_minimization
rm *.trr

# 5- Start NVT equilibration

In [ ]:
%%shell
chmod +x /content/drive/MyDrive/GROMACS_GPU/bin/*
source /content/drive/MyDrive/GROMACS_GPU/bin/GMXRC
cd /content/drive/MyDrive/GROMACS_BIOCODERS/gromacs

gmx grompp -f step4.1_equilibration.mdp -o step4.1_equilibration.tpr -c step4.0_minimization.gro -p topol.top -n index.ndx -maxwarn 1
gmx mdrun -v -deffnm step4.1_equilibration
rm *.trr

# 6- Start NPT equilibration

In [ ]:
%%shell
chmod +x /content/drive/MyDrive/GROMACS_GPU/bin/*
source /content/drive/MyDrive/GROMACS_GPU/bin/GMXRC
cd /content/drive/MyDrive/GROMACS_BIOCODERS/gromacs

gmx grompp -f step4.2_equilibration.mdp -o step4.2_equilibration.tpr -c step4.1_equilibration.gro -t step4.1_equilibration.cpt -p topol.top -n index.ndx -maxwarn 1
gmx mdrun -v -deffnm step4.2_equilibration
rm *.trr


# 7- Start production (each group should perform 20 ns.) <br> Ex: if you are 5 members, the First member should: <br> A- Build GROMACS on his account <br> B- prepare the system (with CHARMM-GUI), minimize, equilibrate and if possible perform 4 ns on his google drive <br> C- share the reults folder and the folder where GROMACS was built with the second member (see videos below) who will continue the next 4 ns. He will NOT repeat the previous steps (will NOT build, prepare, minimize, equilibrate) <br> D- After that, the first person will share the same folders (results and GROMACS) with the third person who will make the third run (from 8 ns to 12 ns) and so on until 20 ns is performed



# For sharing data without downloading the data see this video: https://youtu.be/64UMut7rcmE



# You can build GROMACS with GPU on one account and share the folder with your group as shown in the following video. https://youtu.be/XqwFB67hHRU <br> This way, you can use GPU continously and each member will continue the run from the previous one. <br> use the following cell to run the simulation. change the \<PATH> according to your destination



In [ ]:
%%shell

# this code is for the first run only
chmod +x /content/drive/MyDrive/GROMACS_GPU/bin/*
source /content/drive/MyDrive/GROMACS_GPU/bin/GMXRC
cd /content/drive/MyDrive/computer_vision/gromacs

gmx grompp -f step5_production.mdp -o step5_production.tpr -c step4.2_equilibration.gro -t step4.2_equilibration.cpt -p topol.top -n index.ndx
gmx mdrun -v -nb gpu -deffnm step5_production

rm *.trr

#### Use this code for the next runs to continue from a previous run

In [ ]:
%%shell
chmod +x /content/drive/MyDrive/GROMACS_GPU/bin/*
source /content/drive/MyDrive/GROMACS_GPU/bin/GMXRC
cd /content/drive/MyDrive/computer_vision/gromacs

gmx mdrun -v -deffnm step5_production -noappend -cpi step5_production.cpt
rm *.trr


In [ ]:
!nvidia-smi